# FCFF DCF 결과 분석 v1 (읽기 전용)
배치 노트북을 다시 돌리지 않고 DB `us_fcff_dcf_valuation` 에 저장된 결과만 조회합니다.

**사용법** : Cell 1 실행 후, 필요한 셀만 골라서 실행. 엑셀은 `fa.OUT_DIR` 에 `_v1, _v2 ...` 로 버전이 올라가며 저장됩니다.

`fcff_analysis_v1.py` 는 이 노트북과 같은 폴더(또는 프로젝트 루트)에 두세요.

In [1]:
# ── Cell 1 · 모듈 로드 ─────────────────────────────────────────
import importlib, pandas as pd
import fcff_analysis_v1 as fa
importlib.reload(fa)                      # 모듈 수정 후 재실행 시 반영
from IPython.display import display
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

# (선택) 품질 플래그 기준 조정 — 모듈 기본값 덮어쓰기
# fa.UPSIDE_MAX, fa.UPSIDE_MIN = 300.0, -90.0
# fa.OUT_DIR = fa.Path(r"C:\Users\82108\OneDrive\INVESTMENT\미국주식\FCFF_분석")

print("프로젝트 루트 :", fa.ROOT)
print("엑셀 저장 폴더:", fa.OUT_DIR)

프로젝트 루트 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
엑셀 저장 폴더: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\reports\fcff_analysis


In [2]:
# ── Cell 2 · 저장된 평가일자 ──────────────────────────────────
dates = fa.list_dates()

평가일자 33건 (full_batch = 종목 500개 이상)
      date  n_tickers  n_rows         first_saved          last_saved  full_batch
2026-11-17        955    7640 2026-11-17 22:24:14 2026-11-18 03:20:56        True
2026-11-05        950    7600 2026-11-05 17:44:43 2026-11-05 22:56:38        True
2026-10-06        956    7648 2026-10-06 00:26:55 2026-10-06 14:08:09        True
2026-09-05         50     400 2026-09-05 00:30:55 2026-09-05 00:46:59       False
2026-08-20          8      64 2026-08-20 15:06:34 2026-08-20 21:41:56       False
2026-08-19         52     416 2026-08-19 15:35:44 2026-08-19 16:17:28       False
2026-08-04        952    7616 2026-08-04 11:02:17 2026-08-04 16:11:06        True
2026-07-03        960    7680 2026-07-03 20:35:08 2026-07-04 01:44:26        True
2026-07-02          3      24 2026-07-02 23:32:39 2026-07-02 23:52:21       False
2026-06-30       1616   12928 2026-06-30 21:00:14 2026-07-01 06:18:47        True
2026-06-21          7      56 2026-06-21 11:42:33 2026-06-21 11

In [3]:
# ── Cell 3 · 원클릭 종합 리포트 → 엑셀 ────────────────────────
#   시트: Summary / Top / Bottom / All / Flagged / Compare / Failures
#   DATE=None → 최신 정상 배치, PREV=None → 직전 정상 배치와 비교
DATE = None            # 예) "2026-11-17"  또는 "2026-11-17,2026-11-18"
PREV = None            # 예) "2026-11-05"
path = fa.report(DATE, prev=PREV, top_n=50)
display(fa.summary(DATE, PREV))

[failures] failed_tickers_2026-11-17.txt 를 찾지 못함 (탐색: 작업폴더, 루트, 루트 하위 2단계)
[export] C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\reports\fcff_analysis\FCFF_Report_20261117_v1.xlsx


,item,value
0,평가일자,2026-11-17
1,종목 수,955
2,TP <= 0 (음수 목표가),162
3,Upside 극단값 (<-90% 또는 >300%),237
4,g = GDP 상한(4%),800
5,WACC = 하한(Rf+1%),35
6,WACC-g 스프레드 = 하한(2%),57
7,clean 종목 수,718
8,clean 중 Upside > 0,361
9,clean 중 Upside > 30%,265


In [4]:
# ── Cell 4 · Upside 랭킹 (필터 자유) ──────────────────────────
#   clean=True  : 음수 TP / 극단 upside 제외
#   extra_filter: pandas query 문법 — 사용 가능 컬럼은 결과표 컬럼명 그대로
top = fa.ranking(
    dates=None,
    top_n=30,
    ascending=False,
    clean=True,
    extra_filter="market_cap > 5e9 and not flag_wacc_floor",
)
display(top)
# fa.export_excel({"Rank": top}, "FCFF_Rank_custom")

,rank,ticker,eval_date,current_price,target_price,upside_pct,wacc,g_terminal,tv_spread,reinvestment_rate,market_cap,equity_value,enterprise_value,net_debt,shares,equity_to_mcap,flag_tp_neg,flag_extreme,flag_g_cap,flag_wacc_floor,flag_thin_spread
0,1,GEN,2026-11-17,29.94100,118.837703,296.906260,0.075707,0.040000,0.035707,0.010154,1.805442e+10,7.165914e+10,7.918614e+10,7.527000e+09,6.030000e+08,3.969063,False,False,True,False,False
1,2,AVGO,2026-11-17,333.40000,1274.260005,282.201561,0.093814,0.040000,0.053814,0.040314,1.629326e+12,6.227309e+12,6.262753e+12,3.544400e+10,4.887000e+09,3.822016,False,False,True,False,False
2,3,KDP,2026-11-17,35.58000,135.652962,281.261838,0.061170,0.040000,0.021170,0.147090,4.860584e+10,1.853155e+11,2.128905e+11,2.757500e+10,1.366100e+09,3.812618,False,False,True,False,False
3,4,STRL,2026-11-17,275.27800,1046.814781,280.275496,0.107202,0.040000,0.067202,0.194708,8.553163e+09,3.252558e+10,3.247445e+10,-5.113100e+07,3.107100e+07,3.802755,False,False,True,False,False
4,5,M,2026-11-17,23.62000,89.399019,278.488650,0.077416,0.040000,0.037416,1.212292,6.422278e+09,2.430759e+10,2.810259e+10,3.795000e+09,2.719000e+08,3.784886,False,False,True,False,False
5,6,TMHC,2026-11-17,72.45000,259.361731,257.987206,0.087876,0.040000,0.047876,0.080095,7.066048e+09,2.529555e+10,2.705722e+10,1.761670e+09,9.753000e+07,3.579872,False,False,True,False,False
6,7,PFE,2026-11-17,28.86000,103.214482,257.638536,0.063997,0.040000,0.023997,0.236159,1.659739e+11,5.935865e+11,6.544335e+11,6.084700e+10,5.751000e+09,3.576385,False,False,True,False,False
7,8,LYB,2026-11-17,61.32000,214.467737,249.751691,0.066047,0.040000,0.026047,0.587171,1.980636e+10,6.927308e+10,7.961408e+10,1.034100e+10,3.230000e+08,3.497517,False,False,True,False,False
8,9,LDOS,2026-11-17,129.17000,441.755965,241.995792,0.067715,0.040000,0.027715,0.078654,1.614625e+10,5.521950e+10,6.115350e+10,5.934000e+09,1.250000e+08,3.419958,False,False,True,False,False
9,10,HON,2026-11-17,217.95000,741.676766,240.296750,0.078571,0.040000,0.038571,0.200370,6.956964e+10,2.367432e+11,2.476422e+11,1.089900e+10,3.192000e+08,3.402967,False,False,True,False,False


In [5]:
# ── Cell 5 · 종목 상세 (분기별 FCFF 경로 + 평가 이력) ─────────
TICKERS = ["NVDA", "LLY", "MU"]
for t in TICKERS:
    print(f"\n== {t} 평가 이력 ==")
    display(fa.ticker_history(t))
display(fa.ticker_detail(TICKERS[0]))          # date=None → 가장 최근 평가일
# fa.ticker_report(TICKERS)                    # 엑셀로 저장


== NVDA 평가 이력 ==


,ticker,eval_date,current_price,target_price,upside_pct,wacc,g_terminal,equity_value,shares,tp_chg_pct
0,NVDA,2026-04-09,181.79000,13.984664,-92.325396,0.194291,0.04,3.416760e+11,2.443200e+10,NaN
1,NVDA,2026-04-10,183.91000,98.577110,-46.399266,0.194296,0.04,3.416622e+11,2.443200e+10,6.048944
2,NVDA,2026-04-30,209.25000,1301.432928,522.248591,0.194355,0.04,3.179661e+13,2.443200e+10,12.202182
3,NVDA,2026-06-07,205.10000,539.010088,162.803553,0.112961,0.04,1.314700e+13,2.439100e+10,-0.585833
4,NVDA,2026-06-12,204.87000,538.889204,163.039588,0.112974,0.04,1.314405e+13,2.439100e+10,-0.000224
5,NVDA,2026-06-14,205.19000,471.887231,129.975745,0.112993,0.04,1.150980e+13,2.439100e+10,-0.124333
6,NVDA,2026-06-15,205.19000,471.887231,129.975745,0.112993,0.04,1.150980e+13,2.439100e+10,0.000000
7,NVDA,2026-06-30,194.97000,470.539404,141.339387,0.113169,0.04,1.147693e+13,2.439100e+10,-0.002856
8,NVDA,2026-07-03,194.83000,740.792900,280.225273,0.113225,0.04,1.806868e+13,2.439100e+10,0.574348
9,NVDA,2026-08-04,206.64000,739.605455,257.919790,0.113321,0.04,1.803972e+13,2.439100e+10,-0.001603



== LLY 평가 이력 ==


,ticker,eval_date,current_price,target_price,upside_pct,wacc,g_terminal,equity_value,shares,tp_chg_pct
0,LLY,2026-04-09,950.17999,203.248905,-78.609431,0.099548,0.04,1.825179e+11,898002000.0,NaN
1,LLY,2026-04-13,939.47000,2609.018396,178.489867,0.099499,0.04,2.342340e+12,898002000.0,11.836568
2,LLY,2026-04-14,929.55000,2610.781450,180.865091,0.099453,0.04,2.344487e+12,898002000.0,0.000676
3,LLY,2026-06-07,1131.42000,1599.091460,41.334912,0.070033,0.04,1.432626e+12,895900000.0,-0.387505
4,LLY,2026-06-12,1160.95000,1603.586081,38.127058,0.069956,0.04,1.436653e+12,895900000.0,0.002811
5,LLY,2026-06-14,1133.00000,1596.234559,40.885663,0.070049,0.04,1.430067e+12,895900000.0,-0.004584
6,LLY,2026-06-15,1133.00000,1596.234559,40.885663,0.070049,0.04,1.430067e+12,895900000.0,0.000000
7,LLY,2026-06-21,1098.13000,1594.514900,45.202745,0.070112,0.04,1.428526e+12,895900000.0,-0.001077
8,LLY,2026-06-30,1229.93000,1608.387370,30.770643,0.069840,0.04,1.440954e+12,895900000.0,0.008700
9,LLY,2026-07-03,1210.50000,1605.959551,32.669108,0.069881,0.04,1.438779e+12,895900000.0,-0.001509



== MU 평가 이력 ==


,ticker,eval_date,current_price,target_price,upside_pct,wacc,g_terminal,equity_value,shares,tp_chg_pct
0,MU,2026-04-10,421.5100,1052.013165,149.582018,0.169645,0.04,1.199295e+12,1.140000e+09,NaN
1,MU,2026-04-29,523.8200,627.988425,20.999697,0.170358,0.04,7.159068e+11,1.140000e+09,-0.403060
2,MU,2026-04-30,520.8267,716.362132,38.171148,0.170339,0.04,8.166528e+11,1.140000e+09,0.140725
3,MU,2026-05-01,517.1600,8646.611105,1571.941199,0.105263,0.04,9.857137e+12,1.140000e+09,11.070168
4,MU,2026-05-29,923.5200,8069.686428,773.796607,0.109345,0.04,9.199443e+12,1.140000e+09,-0.066723
5,MU,2026-05-30,971.0000,7980.199215,721.853678,0.110026,0.04,9.097427e+12,1.140000e+09,-0.011089
6,MU,2026-06-06,864.0100,7906.363799,815.077811,0.110597,0.04,9.013255e+12,1.140000e+09,-0.009252
7,MU,2026-06-07,864.0100,7864.662904,810.251375,0.110924,0.04,8.965716e+12,1.140000e+09,-0.005274
8,MU,2026-06-12,995.8700,7928.943974,696.182632,0.110421,0.04,9.038996e+12,1.140000e+09,0.008173
9,MU,2026-06-14,981.6100,1350.516649,37.581794,0.110525,0.04,1.539589e+12,1.140000e+09,-0.829673


,ticker,eval_date,quarter,sales_forecast,opm_forecast,ebit,tax_rate,nopat,da,capex,nwc,delta_nwc,fcff,roic,wacc,g_terminal,target_price,current_price,upside_pct
0,NVDA,2026-11-17,2026Q4,1.126446e+11,0.666908,7.512355e+10,0.21,5.934760e+10,1.175297e+09,2.691803e+09,1.383683e+11,2.017411e+10,3.765699e+10,2.595462,0.113477,0.04,479.746709,156.14,207.254201
1,NVDA,2026-11-17,2027Q1,1.270379e+11,0.681504,8.657686e+10,0.21,6.839572e+10,1.325472e+09,3.035751e+09,1.560485e+11,1.768018e+10,4.900527e+10,3.301601,0.113477,0.04,479.746709,156.14,207.254201
2,NVDA,2026-11-17,2027Q2,1.441147e+11,0.673068,9.699897e+10,0.21,7.662919e+10,1.503646e+09,3.443825e+09,1.770250e+11,2.097647e+10,5.371254e+10,3.137931,0.113477,0.04,479.746709,156.14,207.254201
3,NVDA,2026-11-17,2027Q3,1.606618e+11,0.678384,1.089903e+11,0.21,8.610237e+10,1.676294e+09,3.839242e+09,1.973508e+11,2.032586e+10,6.361356e+10,3.563088,0.113477,0.04,479.746709,156.14,207.254201
4,NVDA,2026-11-17,2027Q4,1.792531e+11,0.694891,1.245613e+11,0.21,9.840344e+10,1.870270e+09,4.283508e+09,2.201877e+11,2.283690e+10,7.315330e+10,3.628391,0.113477,0.04,479.746709,156.14,207.254201
5,NVDA,2026-11-17,2028Q1,1.871870e+11,0.709859,1.328764e+11,0.21,1.049723e+11,1.953049e+09,4.473100e+09,2.299334e+11,9.745709e+09,9.270657e+10,7.382638,0.113477,0.04,479.746709,156.14,207.254201
6,NVDA,2026-11-17,2028Q2,2.065718e+11,0.701166,1.448411e+11,0.21,1.144245e+11,2.155303e+09,4.936326e+09,2.537449e+11,2.381145e+10,8.783202e+10,3.980291,0.113477,0.04,479.746709,156.14,207.254201
7,NVDA,2026-11-17,2028Q3,2.211698e+11,0.706600,1.562787e+11,0.21,1.234602e+11,2.307615e+09,5.285167e+09,2.716766e+11,1.793172e+10,1.025509e+11,5.317688,0.113477,0.04,479.746709,156.14,207.254201


In [6]:
# ── Cell 6 · 두 배치 비교 (TP 변화, 부호 반전, 신규/탈락) ──────
cmp_ = fa.compare_runs("2026-11-05", "2026-11-17")
display(cmp_["status"].value_counts())
display(cmp_[cmp_["tp_sign_flip"]].head(30))                     # TP 부호가 바뀐 종목
display(cmp_.query("status == 'both'").nlargest(20, "tp_chg_pct"))  # TP 급등
# fa.export_excel({"Compare": cmp_}, "FCFF_Compare_20261117")

status
both       944
new         11
dropped      6
Name: count, dtype: int64

,ticker,date_old,date_new,status,current_price_old,current_price_new,price_chg_pct,target_price_old,target_price_new,tp_chg_pct,tp_sign_flip,upside_pct_old,upside_pct_new,d_upside,wacc_old,wacc_new,d_wacc,g_terminal_old,g_terminal_new,clean_old,clean_new
44,HROW,2026-11-05,2026-11-17,both,28.750,29.5900,0.029217,-14.477495,9.778733,NaN,True,-150.356504,-66.952574,83.403930,0.067265,0.067345,0.000080,0.00,0.040000,False,True
46,RDNT,2026-11-05,2026-11-17,both,80.260,75.9153,-0.054133,-19.055142,42.584851,NaN,True,-123.741766,-43.904785,79.836981,0.087802,0.086849,-0.000953,0.04,0.040000,False,True
50,TDC,2026-11-05,2026-11-17,both,34.610,34.6300,0.000578,-20.207117,5.579499,NaN,True,-158.385197,-83.888250,74.496947,0.084265,0.084073,-0.000192,0.04,0.040000,False,True
112,CECO,2026-11-05,2026-11-17,both,57.490,57.4900,0.000000,-10.208161,4.428119,NaN,True,-117.756411,-92.297584,25.458828,0.089325,0.091069,0.001743,0.04,0.040000,False,False
140,DIOD,2026-11-05,2026-11-17,both,79.460,83.7017,0.053382,-5.332428,9.341439,NaN,True,-106.710833,-88.839607,17.871227,0.103761,0.097107,-0.006654,0.04,0.040000,False,True
340,KTOS,2026-11-05,2026-11-17,both,35.960,31.9000,-0.112903,-0.244663,1.145045,NaN,True,-100.680375,-96.410518,4.269858,0.085841,0.086195,0.000354,0.04,0.000000,False,False
829,DAL,2026-11-05,2026-11-17,both,89.240,86.6000,-0.029583,5.327914,-1.362498,-1.255728,True,-94.029680,-101.573324,-7.543644,0.084179,0.083442,-0.000736,0.04,0.040000,False,False
886,LCII,2026-11-05,2026-11-17,both,104.184,98.9400,-0.050334,21.008959,-3.009651,-1.143256,True,-79.834755,-103.041895,-23.207141,0.093677,0.079375,-0.014302,0.04,0.040000,True,False
904,SANM,2026-11-05,2026-11-17,both,143.770,151.4950,0.053732,39.859559,-7.355678,-1.184540,True,-72.275468,-104.855393,-32.579925,0.084877,0.087408,0.002531,0.04,0.040000,True,False
929,EXP,2026-11-05,2026-11-17,both,198.190,178.3200,-0.100257,150.309350,-3.594810,-1.023916,True,-24.158964,-102.015932,-77.856968,0.084223,0.082836,-0.001387,0.04,0.040000,True,False


,ticker,date_old,date_new,status,current_price_old,current_price_new,price_chg_pct,target_price_old,target_price_new,tp_chg_pct,tp_sign_flip,upside_pct_old,upside_pct_new,d_upside,wacc_old,wacc_new,d_wacc,g_terminal_old,g_terminal_new,clean_old,clean_new
40,AMEH,2026-11-05,2026-11-17,both,40.82,40.8200,0.000000,2.125122,39.568076,17.619205,False,-94.793920,-3.066938,91.726983,0.065874,0.079244,0.013370,0.040000,0.040000,False,True
31,PCRX,2026-11-05,2026-11-17,both,36.40,36.4800,0.002198,20.255579,68.208893,2.367413,False,-44.352806,86.976133,131.328939,0.068393,0.069632,0.001239,0.000000,0.040000,True,True
47,CENX,2026-11-05,2026-11-17,both,32.80,33.1850,0.011738,11.233850,36.761698,2.272404,False,-65.750456,10.778055,76.528511,0.118460,0.118310,-0.000150,0.040000,0.040000,True,True
45,AXL,2026-11-05,2026-11-17,both,8.62,8.6200,0.000000,5.422425,12.353942,1.278306,False,-37.094839,43.317196,80.412035,0.109803,0.110150,0.000347,0.040000,0.040000,True,True
20,HALO,2026-11-05,2026-11-17,both,130.34,96.1253,-0.262503,233.355349,521.737184,1.235806,False,79.035867,442.767808,363.731941,0.084558,0.079857,-0.004701,0.040000,0.040000,True,False
71,NHC,2026-11-05,2026-11-17,both,212.40,206.0400,-0.029944,79.810265,170.902929,1.141365,False,-62.424546,-17.053519,45.371026,0.072850,0.067999,-0.004851,0.040000,0.040000,True,True
67,ACM,2026-11-05,2026-11-17,both,68.20,79.9300,0.171994,44.122761,93.217479,1.112685,False,-35.303870,16.623895,51.927765,0.079400,0.093898,0.014498,0.040000,0.040000,True,True
293,CALY,2026-11-05,2026-11-17,both,16.60,16.7250,0.007530,0.881571,1.858238,1.107870,False,-94.689329,-88.889460,5.799869,0.098153,0.097001,-0.001152,0.040000,0.040000,False,True
32,HLIO,2026-11-05,2026-11-17,both,66.95,45.0500,-0.327110,44.884143,89.005112,0.982997,False,-32.958710,97.569616,130.528326,0.094628,0.085431,-0.009198,0.040000,0.040000,True,True
24,PRKS,2026-11-05,2026-11-17,both,34.19,39.9400,0.168178,125.012693,247.237140,0.977696,False,265.641102,519.021382,253.380280,0.111985,0.067946,-0.044039,0.040000,0.040000,True,False


In [7]:
# ── Cell 7 · 실패 종목 (체크포인트 파일) ───────────────────────
display(fa.failures("2026-11-17"))

[failures] failed_tickers_2026-11-17.txt 를 찾지 못함 (탐색: 작업폴더, 루트, 루트 하위 2단계)


,ticker,source_file,saved_in_db


In [8]:
# ── Cell 8 · 자유 SQL (SELECT 만 허용) ─────────────────────────
df = fa.q("""
    SELECT ticker, quarter, sales_forecast, opm_forecast, fcff
    FROM us_fcff_dcf_valuation
    WHERE `date` = %s AND ticker IN ('NVDA', 'AMD')
    ORDER BY ticker, quarter
""", ["2026-11-17"])
display(df)
# fa.export_excel({"Query": df}, "FCFF_Query_NVDA_AMD")

,ticker,quarter,sales_forecast,opm_forecast,fcff
0,AMD,2026Q4,1.404401e+10,0.211602,2.127888e+09
1,AMD,2027Q1,1.459938e+10,0.208154,2.366485e+09
2,AMD,2027Q2,1.550340e+10,0.199796,1.968457e+09
3,AMD,2027Q3,1.641519e+10,0.205949,2.227898e+09
4,AMD,2027Q4,1.713041e+10,0.206639,2.666812e+09
5,AMD,2028Q1,1.785501e+10,0.203927,2.770720e+09
6,AMD,2028Q2,1.916272e+10,0.196265,2.111833e+09
7,AMD,2028Q3,2.025981e+10,0.203075,2.743447e+09
8,NVDA,2026Q4,1.126446e+11,0.666908,3.765699e+10
9,NVDA,2027Q1,1.270379e+11,0.681504,4.900527e+10


In [12]:
OUT_DIR_OVERRIDE = r"C:\Users\82108\OneDrive\INVESTMENT\미국주식\FCFF_분석"

import fcff_analysis_v1 as fa
fa.OUT_DIR = fa.Path(r"C:\Users\82108\OneDrive\INVESTMENT\미국주식\FCFF_valuation_results")

print("저장 폴더:", fa.OUT_DIR)

# 2) 저장된 평가일자 확인 (DB 연결 확인용)
fa.list_dates()

# 3) 데이터 추출 -> 화면 출력
clean = fa.ranking("2026-11-17", top_n=None)          # TP>0, upside -90~300%
print(f"clean 종목 수: {len(clean)}")
display(clean.head(30))

# 4) 엑셀 저장
path = fa.export_excel({"Clean": clean}, "FCFF_clean_20261117")
print("저장 완료:", path)

저장 폴더: C:\Users\82108\OneDrive\INVESTMENT\미국주식\FCFF_valuation_results
평가일자 33건 (full_batch = 종목 500개 이상)
      date  n_tickers  n_rows         first_saved          last_saved  full_batch
2026-11-17        955    7640 2026-11-17 22:24:14 2026-11-18 03:20:56        True
2026-11-05        950    7600 2026-11-05 17:44:43 2026-11-05 22:56:38        True
2026-10-06        956    7648 2026-10-06 00:26:55 2026-10-06 14:08:09        True
2026-09-05         50     400 2026-09-05 00:30:55 2026-09-05 00:46:59       False
2026-08-20          8      64 2026-08-20 15:06:34 2026-08-20 21:41:56       False
2026-08-19         52     416 2026-08-19 15:35:44 2026-08-19 16:17:28       False
2026-08-04        952    7616 2026-08-04 11:02:17 2026-08-04 16:11:06        True
2026-07-03        960    7680 2026-07-03 20:35:08 2026-07-04 01:44:26        True
2026-07-02          3      24 2026-07-02 23:32:39 2026-07-02 23:52:21       False
2026-06-30       1616   12928 2026-06-30 21:00:14 2026-07-01 06:18:47      

,rank,ticker,eval_date,current_price,target_price,upside_pct,wacc,g_terminal,tv_spread,reinvestment_rate,market_cap,equity_value,enterprise_value,net_debt,shares,equity_to_mcap,flag_tp_neg,flag_extreme,flag_g_cap,flag_wacc_floor,flag_thin_spread
0,1,GEN,2026-11-17,29.9410,118.837703,296.906260,0.075707,0.0400,0.035707,0.010154,1.805442e+10,7.165914e+10,7.918614e+10,7.527000e+09,6.030000e+08,3.969063,False,False,True,False,False
1,2,GNK,2026-11-17,24.1000,94.394539,291.678584,0.075552,0.0400,0.035552,0.208641,1.075700e+09,4.213285e+09,4.494611e+09,2.813260e+08,4.463484e+07,3.916786,False,False,True,False,False
2,3,SEAS,2026-11-17,50.3000,193.265573,284.225791,0.111493,0.0400,0.071493,0.546620,2.372550e+09,9.115951e+09,9.234073e+09,1.181220e+08,4.716800e+07,3.842258,False,False,True,False,False
3,4,AVGO,2026-11-17,333.4000,1274.260005,282.201561,0.093814,0.0400,0.053814,0.040314,1.629326e+12,6.227309e+12,6.262753e+12,3.544400e+10,4.887000e+09,3.822016,False,False,True,False,False
4,5,KDP,2026-11-17,35.5800,135.652962,281.261838,0.061170,0.0400,0.021170,0.147090,4.860584e+10,1.853155e+11,2.128905e+11,2.757500e+10,1.366100e+09,3.812618,False,False,True,False,False
5,6,STRL,2026-11-17,275.2780,1046.814781,280.275496,0.107202,0.0400,0.067202,0.194708,8.553163e+09,3.252558e+10,3.247445e+10,-5.113100e+07,3.107100e+07,3.802755,False,False,True,False,False
6,7,M,2026-11-17,23.6200,89.399019,278.488650,0.077416,0.0400,0.037416,1.212292,6.422278e+09,2.430759e+10,2.810259e+10,3.795000e+09,2.719000e+08,3.784886,False,False,True,False,False
7,8,CE,2026-11-17,47.8000,179.862702,276.281802,0.055500,0.0355,0.020000,0.900561,5.267560e+09,1.982087e+10,3.033287e+10,1.051200e+10,1.102000e+08,3.762818,False,False,False,True,True
8,9,SGMS,2026-11-17,58.0700,217.482764,274.518278,0.111138,0.0400,0.071138,0.414272,4.500425e+09,1.685491e+10,1.683091e+10,-2.400000e+07,7.750000e+07,3.745183,False,False,True,False,False
9,10,ATGE,2026-11-17,106.6400,398.045270,273.260756,0.079690,0.0400,0.039690,0.187947,3.686865e+09,1.376162e+10,1.429299e+10,5.313660e+08,3.457300e+07,3.732608,False,False,True,False,False


[export] C:\Users\82108\OneDrive\INVESTMENT\미국주식\FCFF_valuation_results\FCFF_clean_20261117_v1.xlsx
저장 완료: C:\Users\82108\OneDrive\INVESTMENT\미국주식\FCFF_valuation_results\FCFF_clean_20261117_v1.xlsx
